# Setup

## Setup Environment

Run the following command to setup the environment and use the respecitve kernel in Jupyther.

```bash
conda create -n SOS2025 python=3.11 -y
conda activate SOS2025
pip install -r requirements.txt
```

## Load Libraries

In [1]:
# Import Starvers for the provenance documentation. 
# Import the libraries required for the exercise, e.g., pandas, numpy, scipy, matplotlib, seaborn and scikit-learn
import numpy as np
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import plotly.express as px
import datetime
import typing
import requests
import time
import shutil
import json
from starvers.starvers import TripleStoreEngine
from scipy.io import arff
import uuid

ModuleNotFoundError: No module named 'pandas'

## Provenance Configuration

In [ ]:
#executed_by = 'stud-id_12022505'  # Replace the digits after "id_" with your own student ID
executed_by = 'stud-id_01541209'  # Replace the digits after "id_" with your own student ID


# group id for this project
group_id = '25'  # Replace the digits with your group id

# Students working on this notebook
student_a = 'stud-id_12022505'  # Replace the digits after "id_" with student A's student ID
raphael = student_a
student_b = 'stud-id_01541209'  # Replace the digits after "id_" with student B's student ID
anton = student_b

# Roles. Don't change these values.
code_writer_role = 'code_writer'
code_executor_role = 'code_executor'

# according to the forum we switch to the BI endpoints
# get_endpoint = "https://starvers.ec.tuwien.ac.at/SOS2025"
# post_endpoint = "https://starvers.ec.tuwien.ac.at/SOS2025/statements"

get_endpoint = "https://starvers.ec.tuwien.ac.at/BI2025"
post_endpoint = "https://starvers.ec.tuwien.ac.at/BI2025/statements"
engine = TripleStoreEngine(get_endpoint, post_endpoint, skip_connection_test=True)

prefixes = {
    'xsd': 'http://www.w3.org/2001/XMLSchema#',
    'foaf': 'http://xmlns.com/foaf/0.1/',
    'prov': 'http://www.w3.org/ns/prov#',
    'sc': 'https://schema.org/',
    'cr': 'http://mlcommons.org/croissant/',
    'mls': 'http://www.w3.org/ns/mls#',
    'mlso': 'http://w3id.org/mlso',
    'siu': 'https://si-digital-framework.org/SI/units/',
    'siq': 'https://si-digital-framework.org/SI/quantities/',
    'qudt': 'http://qudt.org/schema/qudt/',
    '': f'https://starvers.ec.tuwien.ac.at/SOS2025/{group_id}/',
}

### Registration
ONLY RUN ONCE! And i already did that!

In [ ]:
raise(Exception("Registration triples have already been inserted. Do not run this cell again!

Ontologies used: foaf, prov, IAO
reigstration_triples_a = [
f':{student_a} rdf:type foaf:Person .',
f':{student_a} rdf:type prov:Agent .',
f':{student_a} foaf:givenName "Raphael" .',
f':{student_a} foaf:familyName "Dick" .',
f':{student_a} <http://vivoweb.org/ontology/core#identifier> :{student_a} .',
f':{student_a} rdf:type <http://purl.obolibrary.org/obo/IAO_0000578> .',
f':{student_a} <http://www.w3.org/2000/01/rdf-schema#label> "Immatriculation number" .',
f':{student_a} <http://purl.obolibrary.org/obo/IAO_0000219> "12022505"^^xsd:string .',
]

reigstration_triples_b = [
f':{student_b} rdf:type foaf:Person .',
f':{student_b} rdf:type prov:Agent .',
f':{student_b} foaf:givenName "Anton" .',
f':{student_b} foaf:familyName "Stoiber" .',
f':{student_b} <http://vivoweb.org/ontology/core#identifier> :{student_b} .',
f':{student_b} rdf:type <http://purl.obolibrary.org/obo/IAO_0000578> .',
f':{student_b} <http://www.w3.org/2000/01/rdf-schema#label> "Immatriculation number" .',
f':{student_b} <http://purl.obolibrary.org/obo/IAO_0000219> "01541209"^^xsd:string .',
]

role_triples = [
    f':{code_writer_role} rdf:type prov:Role .',
    f':{code_executor_role} rdf:type prov:Role .',
]

engine.insert(reigstration_triples_a, prefixes=prefixes)
engine.insert(reigstration_triples_b, prefixes=prefixes)
engine.insert(role_triples, prefixes=prefixes)

Exception: ### Registration triples have already been inserted. Do not run this cell again! ###

# Helper Scripts

In [95]:
# GENERATE UUID
# ---
# Execute this cell to generate a new uuid that you can use instead of blank nodes
# for e.g. activities, executors, writers, ...

print(uuid.uuid4())

45b533ae-f03b-4723-8fb0-824b696e7d5e


# Helper Functions

In [87]:
def now() -> str:
    """
    Returns the current time in ISO 8601 format with UTC timezone in the following format:
    YYYY-MM-DDTHH:MM:SS.sssZ
    """
    timestamp = datetime.datetime.now(datetime.timezone.utc)
    timestamp_formated = timestamp.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]  +"Z"

    return timestamp_formated

def executor_triplets(activity_id: str, executor_uuid: str):
	return [
    f':{activity_id} prov:qualifiedAssociation :{executor_uuid} .',
    f':{executor_uuid} prov:agent :{executed_by} .',
    f':{executor_uuid} rdf:type prov:Association .',
    f':{executor_uuid} prov:hadRole :{code_executor_role} .',
	]

def writer_triplets(activity_id: str, writer_id: str, writer_uuid: str, ):
	return [
    f':{activity_id} prov:qualifiedAssociation :{writer_uuid} .',
    f':{writer_uuid} prov:agent :{writer_id} .',
    f':{writer_uuid} rdf:type prov:Association .',
    f':{writer_uuid} prov:hadRole :{code_writer_role} .',
	]

def phase_triplets(phase_id: str, name: str):
	return [
		f':{phase_id} rdf:type prov:Activity .',
		f':{phase_id} rdfs:label "{name}" .', 
	]

def activity_base_triplets(activity_id: str, phase_id: str, name: str, report: str, start_time: str, end_time: str):
	return [
    f':{activity_id} rdf:type prov:Activity .',
    f':{activity_id} sc:isPartOf :{phase_id} .',
    f':{activity_id} rdfs:comment \'{name}\' .',
    f':{activity_id} rdfs:comment """{report}""" .', 
    f':{activity_id} prov:startedAtTime "{start_time}"^^xsd:dateTime .',
    f':{activity_id} prov:endedAtTime "{end_time}"^^xsd:dateTime .',
	]



# Data Understanding

In [ ]:
# Prov
prov__data_understanding_phase = "data_understanding_phase"
engine.insert(phase_triplets(prov__data_understanding_phase, "Data Understanding Phase"), prefixes=prefixes)

## Load Data

In [29]:
# Config
data_path = os.path.join("data", "datasets")

# Functions
def load_arff_data(file_name: str) -> pd.DataFrame:

	### Load your data
	input_file = os.path.join(data_path, file_name)

	# load data from .arff file
	arff_file = arff.loadarff(input_file)
	raw_data = pd.DataFrame(arff_file[0])

	loaded_data = raw_data
	
	return loaded_data


# Execution
start_time = now()
data = load_arff_data("dataset_14_mfeat-fourier.arff")
end_time = now()

In [30]:
# Provenance
# Describe the data loading activity
prov__du__load_data = "du__load_data"

engine.insert([
	*executor_triplets(prov__du__load_data, "f7a8b191-9068-44d5-8b2c-42c510db48fa"),
	*writer_triplets(prov__du__load_data, raphael, "dd355588-c291-4a28-b803-75af3892db87"),
	*activity_base_triplets(
		prov__du__load_data,
		prov__data_understanding_phase,
		"Load Data",
		"""
			Load the dataset obtained from OpenML, from .arff format into a pandas dataframe
		""",
		start_time,
		end_time
	),
	*[ # inputs
    f':{prov__du__load_data} prov:used :raw_data .',
    f':{prov__du__load_data} prov:used :raw_data_path .',
    f':raw_data rdf:type prov:Entity .',
    f':raw_data_path rdf:type prov:Entity .',
    f':raw_data prov:wasDerivedFrom :raw_data_path .',
	],
	*[ # outputs
    f':data rdf:type prov:Entity .',
    f':data prov:wasGeneratedBy :{prov__du__load_data} .',
    f':data prov:wasDerivedFrom :raw_data .',
	]
], prefixes=prefixes)

## Describing the Raw data

In [72]:
def feature_field(i: int):
	return [
		f':raw_recordset cr:field :field_att{i} .',
		f':field_att{i} rdf:type cr:Field .',
		f':field_att{i} sc:name \'Attribute {i}\' .',
		f':field_att{i} sc:description \'This field describes feature {i} of handwritten numberals\' .',
		f':field_att{i} cr:dataType xsd:float .',
	]

engine.insert([
	':raw_data rdf:type sc:Dataset .',
	':raw_data sc:name \'mfeat-fourier\' .',
	':raw_data sc:description \'One of a set of 6 datasets describing features of handwritten numerals (0 - 9) extracted from a collection of Dutch utility maps. Corresponding patterns in different datasets correspond to the same original character. 200 instances per class (for a total of 2,000 instances) have been digitized in binary images.\' .',

	':data_arff rdf:type cr:FileObject .',
	':data_arff sc:name \'dataset_14_mfeat-fourier.arff\' .',
	':data_arff sc:encodingFormat \'text/arff\' .',
	':raw_data sc:distribution :data_arff .',

	':raw_recordset rdf:type cr:RecordSet .',
	':raw_recordset sc:name \'Table of features of handwritter numerals (0-9)\' .',
	':raw_recordset cr:source :data_arff .',
	':raw_data cr:recordSet :raw_recordset .',

	# Class
	':raw_recordset cr:field :field_class .',
	':field_class rdf:type cr:Field .',
	':field_class sc:name \'Class (target)\' .',
	':field_class sc:description \'The target class consisting of 10 distinct nominal values (1 to 10)\' .',
	':field_class cr:dataType xsd:int .',

	# Features (att1 - att76)
	*[field for i in range(1, 77) for field in feature_field(i)],
], prefixes=prefixes)

In [93]:
# Provenance
prov__du__describing_raw_data = "du__describing_raw_data"

start_time = now()
end_time = now()

engine.insert([
	*executor_triplets(prov__du__describing_raw_data, "f7611786-1a2b-407d-a6d4-292e03e927d0"),
	*writer_triplets(prov__du__describing_raw_data, raphael, "1c9c335f-d5c6-41c6-935f-4aa181b1518e"),
	*activity_base_triplets(
		prov__du__describing_raw_data,
		prov__data_understanding_phase,
		"Describe Raw Data",
		"""
			Added provenance information describing the raw data loaded from the .arff file into a pandas dataframe, and related information
		""",
		start_time,
		end_time
	),
	*[ # inputs
    f':{prov__du__describing_raw_data} prov:used :raw_data .',
	]
], prefixes=prefixes)

## Describing the Data

In [73]:
engine.insert([
	':data rdf:type sc:Dataset .',
	':recordset rdf:type cr:RecordSet .',
	':data cr:recordSet :recordset .',

	':recordset cr:field :field_class .',
	*[field for i in range(1, 77) for field in [
		f':recordset cr:field :field_att{i} .'
	]]
], prefixes=prefixes)

In [96]:
# Provenance
prov__du__describing_data = "du__describing_data"

start_time = now()
end_time = now()

engine.insert([
	*executor_triplets(prov__du__describing_data, "bd2e7a4b-9bd0-42fb-b8f6-1408a78c575a"),
	*writer_triplets(prov__du__describing_data, raphael, "45b533ae-f03b-4723-8fb0-824b696e7d5e"),
	*activity_base_triplets(
		prov__du__describing_data,
		prov__data_understanding_phase,
		"Describe Data",
		"""
			Added provenance information describing the data loaded and related information
		""",
		start_time,
		end_time
	),
	*[ # inputs
    f':{prov__du__describing_data} prov:used :data .',
	]
], prefixes=prefixes)

## Extending the descriptions

In [ ]:
def field_characteristics(field_name: str, characteristics: typing.Dict[str, typing.Any]) -> typing.List[str]:
	triplets = [
		f':field_{field_name} cr:hasCharacteristic :field_{field_name}_characteristics .',
		f':field_{field_name}_characteristics rdf:type cr:FieldCharacteristics .',
	]

	for char_name, char_value in characteristics.items():
		char_value_str = str(char_value).replace('\'', '\\\'').replace('\"', '\\"')
		triplets.append(f':field_{field_name}_characteristics cr:{char_name} \'{char_value_str}\' .')

	return triplets

def calculate_feature_characteristics(data: pd.DataFrame, field_name: str) -> typing.Dict[str, typing.Any]:
	characteristics = {}

	series = data[field_name]

	characteristics['minValue'] = series.min()
	characteristics['maxValue'] = series.max()
	characteristics['meanValue'] = series.mean()
	characteristics['standardDeviation'] = series.std()
	characteristics['numberOfMissingValues'] = series.isna().sum()
	characteristics['sparsity'] = series.isna().sum() / len(series)
	characteristics['numberOfUniqueValues'] = series.nunique()
	characteristics['numberOfOutliers'] = ((series < (series.mean() - 3 * series.std())) | (series > (series.mean() + 3 * series.std()))).sum()
	characteristics['correlationWithClass'] = data[[field_name, 'class']].corr().iloc[0,1]

	return characteristics

feature_characteristics_triplets = []

for i in range(1, 77):
	field_name = f'att{i}'
	characteristics = calculate_feature_characteristics(data, field_name)
	field_triplets = field_characteristics(field_name, characteristics)
	feature_characteristics_triplets.extend(field_triplets)

engine.insert(feature_characteristics_triplets, prefixes=prefixes)

In [97]:
# Provenance
prov__du__field_characteristics = "du__field_characteristics"

engine.insert([
	*executor_triplets(prov__du__field_characteristics, "baa60518-50c9-449c-858a-2ec5dd9536c6"),
	*writer_triplets(prov__du__field_characteristics, raphael, "a213895a-179e-454a-8e98-a870a8a3f3cb"),
	*activity_base_triplets(
		prov__du__field_characteristics,
		prov__data_understanding_phase,
		"Extend Field with Field Characteristics",
		"""
			Added provenance information describing the field characteristics of the data, thereby calculating
			- minValue
			- maxValue
			- meanValue
			- standardDeviation
			- numberOfMissingValues (absolute number of missing values)
			- sparsity (proportion of missing values)
			- numberOfUniqueValues (number of unique values)
			- numberOfOutliers (values outside mean +/- 3*std)
			- correlationWithClass (Pearson correlation with the target class)
		""",
		start_time,
		end_time
	),
	*[ # inputs
    f':{prov__du__describing_data} prov:used :data .',
	]
], prefixes=prefixes)

## Missing

what metrics are still missing: 
- size (the overall dataset size) - i guess we need to add a single triplet to the :data or :recordset 
- maybe more infos regarding size, sparsity, outliers, missing values, correlations (with respect to the whole data set, idk if its enough what i did already)
- hypotheses you might have concerning the distribution of the data, number of clusters and their relationship, majority/minority classes as rdf comment field in the provenance graph

In [ ]:
# Dataset/recordset size (overall rows/cols)
engine.insert([
    ':recordset cr:size "2000"^^xsd:int .',      # total rows
    ':data sc:numberOfVariables "77"^^xsd:int .',# features + class
], prefixes=prefixes)


In [ ]:
start_time = now()
rows, cols = data.shape
overall_missing = int(data.isna().sum().sum())
overall_sparsity = overall_missing / (rows * cols)
class_counts = data['class'].value_counts().to_dict()
hypothesis = """Expect ~10 clusters (digits 0-9); some overlap among similar digits; class balance per counts above."""
end_time = now()

activity_id = "du__dataset_characteristics"
engine.insert([
    *executor_triplets(activity_id, executed_by),
    *writer_triplets(activity_id, executed_by, activity_id),
    *activity_base_triplets(
        activity_id,
        prov__data_understanding_phase,
        "Dataset-level characteristics",
        f"rows={rows}, cols={cols}, missing={overall_missing}, sparsity={overall_sparsity:.4f}, class_dist={class_counts}",
        start_time,
        end_time
    ),
    f':{activity_id} prov:used :data .',
    ':data sc:comment """' + hypothesis + '""" .'
], prefixes=prefixes)


In [ ]:
engine.insert([
    f':data sc:additionalProperty "overall_missing={overall_missing}, overall_sparsity={overall_sparsity:.4f}" .'
], prefixes=prefixes)


# Data Preparation

In [ ]:
prov__data_preparation_phase = "data_preparation_phase"
engine.insert(phase_triplets(prov__data_preparation_phase, "Data Preparation Phase"), prefixes=prefixes)

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- preprocess
start_time = now()
X = data.drop(columns=['class']).values   # features
y = data['class'].values                  # labels (kept unchanged)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
end_time = now()


In [ ]:
# --- provenance
activity_id = "dp__scaling_standard"  # unique ID for this prep step
engine.insert([
    *executor_triplets(activity_id, executed_by),
    *writer_triplets(activity_id, executed_by, activity_id),
    *activity_base_triplets(
        activity_id,
        prov__data_preparation_phase,
        "Standardize features",
        "StandardScaler (mean=0, std=1) on 76 features; class unchanged.",
        start_time,
        end_time
    ),
    f':{activity_id} prov:used :data .',
    ':scaled_data rdf:type sc:Dataset .',
    ':scaled_data sc:name "mfeat-fourier scaled" .',
    ':scaled_data cr:recordSet :recordset_scaled .',
    f':{activity_id} prov:generated :scaled_data .'
], prefixes=prefixes)